In [ ]:
%python -m spacy download fr_core_news_lg #télécharge le modèle

In [2]:
import pickle
import ast
import json
import time
import requests
from datetime import datetime
from tqdm import tqdm
import pandas as pd
import numpy as np

In [3]:
pd.set_option('display.max_columns', 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [4]:
import os
import s3fs

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

In [5]:
with fs.open(r'arnaultchatelain/rap_project/other_scandal_detection/presse/20260130.fmdb_16_22.feather', mode='rb') as f:
    presse = pd.read_feather(f)

In [6]:
presse = presse.drop_duplicates('titre_text_cut').reset_index(drop=True)

# General NER retrieval

In [13]:
nlp = spacy.load('fr_core_news_lg')

In [17]:
docs = nlp.pipe(
    zip(presse.titre_text_cut, presse.parag_idx),
    as_tuples=True,
    disable=['tok2vec', 'parser', 'lemmatizer'],
    batch_size=20,
    n_process=15
)

In [18]:
%%time
counter = 300000
rslt = []
with tqdm(total=len(presse)) as pbar:
    
    for doc, idx in docs: 
        
        for ent in doc.ents:
            if ent.label_ in ['LOC']:
                continue
            else:
                rslt.append((idx, ent.text, ent.label_))
        
        pbar.update(1)
        
        counter += 1 
        if counter % 100000 == 0:
            
            ner_df = pd.DataFrame(rslt, columns=['parag_idx', 'name', 'ner_type'])

            with open(rf'./ner_results/ner_df_{counter}.feather', mode='wb') as f:
                ner_df.to_feather(f)

100%|██████████| 441258/441258 [47:16<00:00, 155.54it/s] 

CPU times: user 46min 41s, sys: 1min, total: 47min 41s
Wall time: 47min 16s


It broke down on sspcloud had to run it twice

In [26]:
ner_df_end = pd.DataFrame(rslt, columns=['parag_idx', 'name', 'ner_type'])

In [27]:
with open(r'ner_df_end.feather', mode='wb') as f:
    ner_df_end.to_feather(f)

In [23]:
with open(r'./ner_results/ner_df_300000.feather', mode='rb') as f:
    ner_df_begin = pd.read_feather(f)

In [28]:
ner_df = pd.concat([ner_df_begin, ner_df_end], ignore_index=True)

In [29]:
with fs.open(r'arnaultchatelain/rap_project/other_scandal_detection/ner_df.feather', mode='wb') as f:
    ner_df.to_feather(f)

In [12]:
with fs.open(r'arnaultchatelain/rap_project/scandal_detection/20260204.ner_df_complete_press.feather', mode='wb') as f:
    ner_df.to_feather(f)